In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_PROPERTIES_PATH = "/Volumes/property_catalog/property_management/landing_zone/properties.csv"
SOURCE_TENANTS_PATH = "/Volumes/property_catalog/property_management/landing_zone/tenants.csv"
SOURCE_RENT_PAYMENTS_PATH = "/Volumes/property_catalog/property_management/landing_zone/rent_payments.csv"
SOURCE_EXPENSES_PATH = "/Volumes/property_catalog/property_management/landing_zone/expenses.csv"
TARGET_SCHEMA = "property_catalog.property_management"

PROPERTY_OCCUPANCY_TABLE = f"{TARGET_SCHEMA}.property_occupancy"
PROPERTY_AVERAGE_LEASE_TABLE = (f"{TARGET_SCHEMA}.property_average_lease")
PROPERTY_AVERAGE_RENT_PAYMENT_TABLE = (f"{TARGET_SCHEMA}.property_average_rent_payment")
PROPERTY_AVERAGE_EXPENSE_TABLE = (f"{TARGET_SCHEMA}.property_average_expense")

properties_df = spark.table(f"{TARGET_SCHEMA}.clean_properties")
tenants_df = spark.table(f"{TARGET_SCHEMA}.clean_tenants")
rent_payments_df = spark.table(f"{TARGET_SCHEMA}.clean_rent_payments")
expenses_df = spark.table(f"{TARGET_SCHEMA}.clean_expenses")

In [0]:
total_properties = properties_df.select("Id").distinct().count()

property_occupancy_df = (
    properties_df
    .groupBy("Status")
    .agg(F.countDistinct("Id").alias("property_count"))
    .withColumn(
        "occupancy_percentage",
        F.round(F.col("property_count") * 100.0 / total_properties, 2)
    )
    .orderBy("Status")
)

# Save property occupancy to a managed Delta table
(
    property_occupancy_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PROPERTY_OCCUPANCY_TABLE)
)

spark.table(PROPERTY_OCCUPANCY_TABLE).show()

In [0]:
property_average_lease_df = (
    tenants_df
    .withColumn(
        "lease_end_date",
        F.coalesce(F.col("LeaseEndDate"), F.current_date())
    )
    .withColumn(
        "lease_duration_days",
        F.datediff(F.col("lease_end_date"), F.col("LeaseStartDate"))
    )
    .groupBy("PropertyId")
    .agg(
        F.round(F.avg("lease_duration_days"), 0).alias("average_lease_days"),
        F.round(F.avg("lease_duration_days") / 365.25, 2).alias("average_lease_years"),
        F.count("*").alias("tenant_count")
    )
    .orderBy(F.desc("average_lease_days"))
)

# Save property average lease to a managed Delta table
(
    property_average_lease_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PROPERTY_AVERAGE_LEASE_TABLE)
)

spark.table(PROPERTY_AVERAGE_LEASE_TABLE).show()

In [0]:
property_average_rent_payments_df = (
    rent_payments_df
    .groupBy("PropertyId")
    .agg(
        F.round(F.avg("Amount"), 2).alias("average_rent_payment"),
    )
    .orderBy(F.desc("average_rent_payment"))
)

# Save property average rent payments to a managed Delta table
(
    property_average_rent_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PROPERTY_AVERAGE_RENT_PAYMENT_TABLE)
)

spark.table(PROPERTY_AVERAGE_RENT_PAYMENT_TABLE).show()

In [0]:
property_average_expenses_df = (
    expenses_df
    .groupBy("PropertyId")
    .agg(
        F.round(F.avg("Amount"), 2).alias("average_expense"),
        F.count("*").alias("expense_count"),
        F.round(F.sum("Amount"), 2).alias("total_expenses")
    )
    .orderBy(F.desc("average_expense"))
)

# Save property average expenses to a managed Delta table
(
    property_average_expenses_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(PROPERTY_AVERAGE_EXPENSE_TABLE)
)

spark.table(PROPERTY_AVERAGE_EXPENSE_TABLE).show()